# Notebook 4: Benchmarking — HumanEval & MBPP (All 4 Conditions)

**Goal:** Measure Pass@1 for **all four conditions** on HumanEval (164 problems) and MBPP (500 problems, test split).

| # | Condition | Method |
|---|---|---|
| 1 | `baseline_generic` | TinyLlama (generic) + CodeLlama spec decoding |
| 2 | `domain_tuned` | TinyLlama (QLoRA-tuned) + CodeLlama spec decoding |
| 3 | `medusa` | CodeLlama + Medusa heads |
| 4 | `eagle2` | CodeLlama + EAGLE-2 draft model |

**Memory strategy:** Each condition is loaded, benchmarked, then deleted before loading the next. This keeps VRAM usage under ~18GB.

---
**Hardware:** A100 (40GB GPU)

## Actual Runtimes (measured on A100)

| Condition | HumanEval | MBPP | Total |
|---|---|---|---|
| C1: baseline_generic | ~29 min (10.5s/problem) | ~82 min (10.5s/problem) | ~1.8 hrs |
| C2: domain_tuned | ~57 min (21.1s/problem) | ~3 hrs 15 min (23.5s/problem) | ~4.2 hrs |
| C3: medusa | ~57 min | ~3 hrs | ~4.0 hrs |
| C4: eagle2 | 28 min 46 sec (exact) | ~82 min | ~1.8 hrs |
| **Total** | | | **~11.8 hrs** |

**Note:** C2 is slower because loading the LoRA adapter + merging weights adds overhead per forward pass. C1/C4 are faster because they use lighter models.

In [ ]:
!pip install -q "transformers==4.44.2" "peft==0.13.2" bitsandbytes accelerate datasets
!pip install -q human-eval evalplus

In [ ]:
import torch
assert torch.cuda.is_available(), "Need GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
import os, json, shutil

drive.mount('/content/drive', force_remount=True)
assert os.path.exists('/content/drive/MyDrive'), "Drive mount failed!"
print("Drive mounted.")

from huggingface_hub import login, hf_hub_download
login()

import wandb
wandb.login()

# ── Model paths (all from HuggingFace Hub) ───────────────────────────────────
TARGET_MODEL_ID = "codellama/CodeLlama-7b-hf"
DRAFT_MODEL_ID  = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
LORA_ADAPTER    = "nishant-k/tinyllama-code-specdraft-100pct"
HF_MEDUSA_REPO  = "nishant-k/medusa-heads-codellama"

# Download medusa heads from HF Hub
MEDUSA_HEADS_LOCAL = "/content/medusa_heads.pt"
if not os.path.exists(MEDUSA_HEADS_LOCAL):
    print("Downloading medusa_heads.pt from HF Hub...")
    MEDUSA_HEADS_LOCAL = hf_hub_download(
        repo_id=HF_MEDUSA_REPO, filename="medusa_heads.pt", local_dir="/content"
    )
    print(f"Downloaded to {MEDUSA_HEADS_LOCAL}")
else:
    print(f"medusa_heads.pt already exists at {MEDUSA_HEADS_LOCAL}")

# ── Benchmark settings ────────────────────────────────────────────────────────
GAMMA            = 5
MAX_NEW_TOKENS   = 256
TEMPERATURE      = 0.8
N_SAMPLES        = 1        # Pass@1 only
VOCAB_SIZE_DRAFT = 32000    # TinyLlama vocab — mask CodeLlama tokens beyond this

RESULTS_DIR   = "./results"
DRIVE_RESULTS = "/content/drive/MyDrive/speculative-decoding-results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

print(f"N_SAMPLES      : {N_SAMPLES} (Pass@1 only)")
print(f"MAX_NEW_TOKENS : {MAX_NEW_TOKENS}")
print(f"Medusa heads   : {MEDUSA_HEADS_LOCAL}")

# ── PEFT unknown-key patch — strips any unrecognised kwargs before __init__ ──
import inspect
from peft import config as _peft_config_module
from peft.config import PeftConfigMixin

if not getattr(PeftConfigMixin.from_peft_type, "_is_stripped_patch", False):
    _orig_from_peft_type = PeftConfigMixin.from_peft_type.__func__

    @classmethod
    def _patched_from_peft_type(cls, **kwargs):
        peft_type = kwargs.get("peft_type", None)
        try:
            config_cls = _peft_config_module.PEFT_TYPE_TO_CONFIG_MAPPING.get(peft_type, cls)
        except Exception:
            config_cls = cls
        valid = set(inspect.signature(config_cls.__init__).parameters.keys()) - {"self"}
        cleaned = {k: v for k, v in kwargs.items() if k in valid}
        dropped = set(kwargs) - set(cleaned)
        if dropped:
            print(f"  [PEFT patch] dropped unknown keys: {dropped}")
        return _orig_from_peft_type(cls, **cleaned)

    _patched_from_peft_type._is_stripped_patch = True
    PeftConfigMixin.from_peft_type = _patched_from_peft_type
    print("PEFT unknown-key patch applied.")
else:
    print("PEFT unknown-key patch already applied — skipping.")

## 1. Shared Utilities — Pass@k, MBPP Executor

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json, time, signal, shutil
from collections import defaultdict
from scipy.special import comb
from huggingface_hub import HfApi

HF_RESULTS_REPO = "nishant-k/speculative-decoding-benchmark-results"

def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased Pass@k estimator (Chen et al., 2021)."""
    if n - c < k:
        return 1.0
    return 1.0 - float(comb(n - c, k, exact=True)) / float(comb(n, k, exact=True))

def compute_pass_at_k(results: list, k_values=(1,)) -> dict:
    task_res = defaultdict(list)
    for r in results:
        task_res[r["task_id"]].append(r["passed"])
    metrics = {}
    for k in k_values:
        scores = [pass_at_k(len(v), sum(v), k) for v in task_res.values() if len(v) >= k]
        metrics[f"pass@{k}"] = float(np.mean(scores)) if scores else 0.0
    return metrics

def run_mbpp_test(completion: str, test_list: list, timeout: float = 10.0) -> bool:
    def handler(signum, frame): raise TimeoutError()
    code = completion + "\n" + "\n".join(test_list)
    # mock input() so problems that call input() never block execution
    safe_globals = {"__builtins__": __builtins__, "input": lambda *a, **k: "0"}
    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(int(timeout))
        exec(compile(code, "<string>", "exec"), safe_globals)
        signal.alarm(0)
        return True
    except Exception:
        return False
    finally:
        signal.alarm(0)

def save_condition_results(condition, humaneval_res, mbpp_res):
    fname = f"benchmark_{condition}.json"
    local_path = os.path.join(RESULTS_DIR, fname)
    with open(local_path, "w") as f:
        json.dump({"humaneval": humaneval_res, "mbpp": mbpp_res}, f, indent=2)
    shutil.copy(local_path, DRIVE_RESULTS)
    print(f"  Saved to Drive: {fname}")
    api = HfApi()
    try:
        api.create_repo(HF_RESULTS_REPO, repo_type="dataset", exist_ok=True, private=False)
    except Exception:
        pass
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=fname,
        repo_id=HF_RESULTS_REPO,
        repo_type="dataset",
    )
    print(f"  Pushed to HF Hub: {HF_RESULTS_REPO}/{fname}")

## 2. Load Benchmark Problems (once, shared across all conditions)

In [ ]:
from datasets import load_dataset

humaneval_problems = list(load_dataset("openai/openai_humaneval", split="test"))
mbpp_problems      = list(load_dataset("google-research-datasets/mbpp", split="test"))

print(f"HumanEval : {len(humaneval_problems)} problems")
print(f"MBPP      : {len(mbpp_problems)} problems")
print(f"Samples per problem: {N_SAMPLES}")

## 3. Benchmark Runner
One function that runs HumanEval + MBPP given any `generate_fn` callable.

In [ ]:
from human_eval.execution import check_correctness
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import wandb

CHECKPOINT_EVERY = 20   # save partial results to Drive every N problems


def _ckpt_path(condition, suite):
    return os.path.join(DRIVE_RESULTS, f"ckpt_{condition}_{suite}.json")

def _load_checkpoint(condition, suite):
    path = _ckpt_path(condition, suite)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"  Resuming {condition}/{suite} from checkpoint: {data['n_done']} problems done")
        return data["results"], set(data["done_task_ids"])
    return [], set()

def _save_checkpoint(condition, suite, results, done_task_ids):
    with open(_ckpt_path(condition, suite), "w") as f:
        json.dump({"results": results, "done_task_ids": list(done_task_ids),
                   "n_done": len(done_task_ids)}, f)

def _clear_checkpoint(condition, suite):
    path = _ckpt_path(condition, suite)
    if os.path.exists(path):
        os.remove(path)


def run_humaneval(generate_fn, condition: str) -> dict:
    print(f"\n[HumanEval] Condition: {condition}")
    results, done_ids = _load_checkpoint(condition, "humaneval")
    n_total = len(humaneval_problems)

    run = wandb.init(
        project="speculative-decoding-code-llm",
        name=f"nb04-humaneval-{condition}",
        config={"condition": condition, "suite": "humaneval",
                "n_problems": n_total, "n_samples": N_SAMPLES,
                "gamma": GAMMA, "temperature": TEMPERATURE},
        reinit=True,
    )

    running_pass = 0
    for i, problem in enumerate(tqdm(humaneval_problems)):
        if problem["task_id"] in done_ids:
            running_pass += int(any(
                r["passed"] for r in results if r["task_id"] == problem["task_id"]
            ))
            continue
        for _ in range(N_SAMPLES):
            completion = generate_fn(problem["prompt"])
            check = check_correctness(problem, completion, timeout=10.0)
            results.append({"task_id": problem["task_id"],
                             "completion": completion, "passed": check["passed"]})
        passed = results[-1]["passed"]
        running_pass += int(passed)
        done_ids.add(problem["task_id"])

        wandb.log({
            "humaneval/running_pass_rate": running_pass / len(done_ids),
            "humaneval/problems_done":     len(done_ids),
            "humaneval/passed_cumulative": running_pass,
        }, step=len(done_ids))

        if (i + 1) % CHECKPOINT_EVERY == 0:
            _save_checkpoint(condition, "humaneval", results, done_ids)
            print(f"  [checkpoint] {len(done_ids)}/{n_total} → Drive")

    metrics = compute_pass_at_k(results, k_values=(1,))
    wandb.log({"humaneval/final_pass@1": metrics["pass@1"]})
    wandb.finish()
    print(f"  Pass@1 : {metrics['pass@1']*100:.2f}%")
    _clear_checkpoint(condition, "humaneval")
    return {"metrics": metrics, "results": results}


def run_mbpp(generate_fn, condition: str) -> dict:
    print(f"\n[MBPP] Condition: {condition}")
    results, done_ids = _load_checkpoint(condition, "mbpp")
    n_total = len(mbpp_problems)

    run = wandb.init(
        project="speculative-decoding-code-llm",
        name=f"nb04-mbpp-{condition}",
        config={"condition": condition, "suite": "mbpp",
                "n_problems": n_total, "n_samples": N_SAMPLES,
                "gamma": GAMMA, "temperature": TEMPERATURE},
        reinit=True,
    )

    running_pass = 0
    for i, problem in enumerate(tqdm(mbpp_problems)):
        tid = str(problem["task_id"])
        if tid in done_ids:
            running_pass += int(any(r["passed"] for r in results if r["task_id"] == tid))
            continue
        prompt = f"# {problem['text']}\n"
        for _ in range(N_SAMPLES):
            completion = generate_fn(prompt)
            passed = run_mbpp_test(completion, problem["test_list"])
            results.append({"task_id": tid, "passed": passed})
        passed = results[-1]["passed"]
        running_pass += int(passed)
        done_ids.add(tid)

        wandb.log({
            "mbpp/running_pass_rate": running_pass / len(done_ids),
            "mbpp/problems_done":     len(done_ids),
            "mbpp/passed_cumulative": running_pass,
        }, step=len(done_ids))

        if (i + 1) % CHECKPOINT_EVERY == 0:
            _save_checkpoint(condition, "mbpp", results, done_ids)
            print(f"  [checkpoint] {len(done_ids)}/{n_total} → Drive")

    metrics = compute_pass_at_k(results, k_values=(1,))
    wandb.log({"mbpp/final_pass@1": metrics["pass@1"]})
    wandb.finish()
    print(f"  Pass@1 : {metrics['pass@1']*100:.2f}%")
    _clear_checkpoint(condition, "mbpp")
    return {"metrics": metrics, "results": results}

## 4. Condition 1 — Baseline Generic

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F

# ── Always load tokenizer + target (needed by C2/C3/C4 too) ──────────────────
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_ID)
target = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
target.eval()
print(f"Target loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")


@torch.no_grad()
def generate_spec(prompt, draft_model):
    device    = next(target.parameters()).device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    tokens_gen = 0

    while tokens_gen < MAX_NEW_TOKENS:
        draft_ids, draft_probs = [], []
        ctx = generated.clone()
        for _ in range(GAMMA):
            logits = draft_model(ctx).logits[:, -1, :] / TEMPERATURE
            probs  = F.softmax(logits, dim=-1)
            token  = torch.multinomial(probs, 1)
            draft_ids.append(token)
            draft_probs.append(probs[0, token.item()].item())
            ctx = torch.cat([ctx, token], dim=-1)

        draft_seq  = torch.cat(draft_ids, dim=-1)
        full_ctx   = torch.cat([generated, draft_seq], dim=-1)

        tgt_logits_raw = target(full_ctx).logits[:, generated.shape[1]-1:-1, :] / TEMPERATURE
        tgt_logits_raw[:, :, VOCAB_SIZE_DRAFT:] = float('-inf')
        tgt_probs = F.softmax(tgt_logits_raw, dim=-1)

        for i in range(GAMMA):
            tok = draft_seq[0, i].item()
            p, q = tgt_probs[0, i, tok].item(), draft_probs[i]
            if torch.rand(1).item() <= min(1.0, p / (q + 1e-8)):
                generated = torch.cat([generated, draft_seq[:, i:i+1]], dim=-1)
                tokens_gen += 1
                if tok == tokenizer.eos_token_id or tokens_gen >= MAX_NEW_TOKENS:
                    break
            else:
                tgt_last_raw = target(generated).logits[:, -1, :] / TEMPERATURE
                tgt_last_raw[:, VOCAB_SIZE_DRAFT:] = float('-inf')
                tgt_last = F.softmax(tgt_last_raw, dim=-1)[0]
                corrected = F.relu(tgt_probs[0, i] - tgt_last)
                mass = corrected.sum()
                if mass < 1e-6:
                    corrected = tgt_probs[0, i]
                else:
                    corrected = corrected / mass
                resampled = torch.multinomial(corrected, 1)         # (1,)
                generated = torch.cat([generated, resampled.unsqueeze(0)], dim=-1)
                tokens_gen += 1
                break

        if tokens_gen >= MAX_NEW_TOKENS:
            break

    return tokenizer.decode(generated[0][input_ids.shape[1]:], skip_special_tokens=True)


# ── Skip C1 benchmark if already done ────────────────────────────────────────
_c1_done_path = os.path.join(DRIVE_RESULTS, "benchmark_baseline_generic.json")
if os.path.exists(_c1_done_path):
    print("baseline_generic already benchmarked — loading from Drive.")
    with open(_c1_done_path) as _f:
        _c1_data = json.load(_f)
    he_c1   = _c1_data["humaneval"]
    mbpp_c1 = _c1_data["mbpp"]
    print(f"  HumanEval Pass@1: {he_c1['metrics']['pass@1']*100:.2f}%")
    print(f"  MBPP      Pass@1: {mbpp_c1['metrics']['pass@1']*100:.2f}%")
else:
    draft_generic = AutoModelForCausalLM.from_pretrained(
        DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
    )
    draft_generic.eval()
    print(f"Both loaded   | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    he_c1   = run_humaneval(lambda p: generate_spec(p, draft_generic), "baseline_generic")
    mbpp_c1 = run_mbpp(lambda p: generate_spec(p, draft_generic),      "baseline_generic")
    save_condition_results("baseline_generic", he_c1, mbpp_c1)

    del draft_generic
    torch.cuda.empty_cache()

## 5. Condition 2 — Domain-Tuned (QLoRA)

In [ ]:
from peft import PeftModel, PeftConfig

# Note: PEFT key stripping already applied at setup (config cell) — do NOT re-patch here

base2       = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
draft_tuned = PeftModel.from_pretrained(base2, LORA_ADAPTER)
draft_tuned.eval()
print(f"Domain-tuned draft loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

he_c2   = run_humaneval(lambda p: generate_spec(p, draft_tuned), "domain_tuned")
mbpp_c2 = run_mbpp(lambda p: generate_spec(p, draft_tuned),      "domain_tuned")
save_condition_results("domain_tuned", he_c2, mbpp_c2)

del draft_tuned, base2
torch.cuda.empty_cache()

## 6. Condition 3 — Medusa

Medusa uses a single CodeLlama forward pass + K extra heads. No separate draft model.

In [ ]:
# ── Inline MedusaHeads definition ────────────────────────────────────────────
class MedusaHeads(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int, num_heads: int = 4):
        super().__init__()
        self.num_heads = num_heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_size, hidden_size, bias=False),
                nn.SiLU(),
                nn.Linear(hidden_size, vocab_size, bias=False),
            )
            for _ in range(num_heads)
        ])

    def forward(self, hidden_states):
        return [head(hidden_states) for head in self.heads]


# Load CodeLlama float16 (del previous target first to free VRAM)
del target; torch.cuda.empty_cache()
target_m = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
target_m.eval()

ckpt  = torch.load(MEDUSA_HEADS_LOCAL, map_location="cpu")
heads = MedusaHeads(ckpt["hidden_size"], ckpt["vocab_size"], ckpt["num_heads"])
heads.load_state_dict(ckpt["state_dict"])
heads = heads.to(next(target_m.parameters()).device).half().eval()
print(f"Medusa loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

NUM_HEADS = ckpt["num_heads"]

@torch.no_grad()
def generate_medusa(prompt: str) -> str:
    device    = next(target_m.parameters()).device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    tokens_gen = 0

    while tokens_gen < MAX_NEW_TOKENS:
        out    = target_m(generated, output_hidden_states=True)
        hidden = out.hidden_states[-1][:, -1:, :].half()    # (1, 1, H) — match heads dtype

        base_token   = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)  # (1, 1)
        head_logits  = heads(hidden)
        draft_tokens = [torch.argmax(hl[:, 0, :], dim=-1, keepdim=True) for hl in head_logits]

        candidate     = torch.cat([base_token] + draft_tokens, dim=-1)
        verify_ids    = torch.cat([generated, candidate], dim=-1)
        verify_logits = target_m(verify_ids).logits
        verify_start  = generated.shape[1]

        generated  = torch.cat([generated, base_token], dim=-1)
        tokens_gen += 1

        for k in range(NUM_HEADS):
            if tokens_gen >= MAX_NEW_TOKENS:
                break
            v_tok = torch.argmax(verify_logits[:, verify_start + k, :], dim=-1).item()
            if v_tok == draft_tokens[k].item():
                generated  = torch.cat([generated, draft_tokens[k]], dim=-1)
                tokens_gen += 1
            else:
                resampled = torch.multinomial(
                    F.softmax(verify_logits[:, verify_start + k, :] / TEMPERATURE, dim=-1), 1
                )  # (1, 1)
                generated  = torch.cat([generated, resampled], dim=-1)
                tokens_gen += 1
                break

    return tokenizer.decode(generated[0][input_ids.shape[1]:], skip_special_tokens=True)


he_c3   = run_humaneval(generate_medusa, "medusa")
mbpp_c3 = run_mbpp(generate_medusa,      "medusa")
save_condition_results("medusa", he_c3, mbpp_c3)

del target_m, heads
torch.cuda.empty_cache()

## 7. Condition 4 — EAGLE-2

In [ ]:
# ── Inline EAGLEDraftModel ────────────────────────────────────────────────────
class EAGLEDraftModel(nn.Module):
    """Simplified EAGLE draft model (untrained — lower bound baseline)."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size * 2),
            nn.SiLU(),
            nn.Linear(hidden_size * 2, hidden_size),
        )

    def forward(self, hidden_state, token_embed):
        return self.fc(torch.cat([hidden_state, token_embed], dim=-1))


base_e = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
base_e.eval()
tokenizer_e = AutoTokenizer.from_pretrained(TARGET_MODEL_ID)

device_e    = next(base_e.parameters()).device
hidden_size = base_e.config.hidden_size
draft_e     = EAGLEDraftModel(hidden_size).half().to(device_e).eval()
print(f"EAGLE (untrained) loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")


@torch.no_grad()
def generate_eagle(prompt: str) -> str:
    embed_layer = base_e.get_input_embeddings()
    input_ids   = tokenizer_e.encode(prompt, return_tensors="pt").to(device_e)
    generated   = input_ids.clone()
    tokens_gen  = 0

    while tokens_gen < MAX_NEW_TOKENS:
        out          = base_e(generated, output_hidden_states=True)
        hidden_state = out.hidden_states[-1][:, -1, :]          # (1, H) float16
        base_probs   = F.softmax(out.logits[:, -1, :] / TEMPERATURE, dim=-1)
        base_token   = torch.multinomial(base_probs, 1)          # (1, 1)

        draft_ids, draft_probs_list = [], []
        cur_hidden = hidden_state
        cur_token  = base_token.squeeze(0)

        for _ in range(GAMMA):
            token_embed = embed_layer(cur_token.unsqueeze(0)).squeeze(1)  # float16
            pred_hidden = draft_e(cur_hidden, token_embed)                 # float16
            d_logits    = base_e.lm_head(pred_hidden)
            d_logits[:, VOCAB_SIZE_DRAFT:] = float('-inf')                # vocab mask
            d_probs     = F.softmax(d_logits / TEMPERATURE, dim=-1)
            d_tok       = torch.multinomial(d_probs, 1).squeeze(0)
            draft_ids.append(d_tok.unsqueeze(0))
            draft_probs_list.append(d_probs[0, d_tok.item()].item())
            cur_hidden = pred_hidden
            cur_token  = d_tok

        draft_seq  = torch.cat(draft_ids, dim=-1)
        candidate  = torch.cat([base_token, draft_seq], dim=-1)
        full_ids   = torch.cat([generated, candidate], dim=-1)
        tgt_logits_raw = base_e(full_ids).logits[:, generated.shape[1]-1:-1, :] / TEMPERATURE
        tgt_logits_raw[:, :, VOCAB_SIZE_DRAFT:] = float('-inf')
        tgt_probs  = F.softmax(tgt_logits_raw, dim=-1)

        generated  = torch.cat([generated, base_token], dim=-1)
        tokens_gen += 1

        for i in range(GAMMA):
            if tokens_gen >= MAX_NEW_TOKENS:
                break
            tok = draft_seq[0, i].item()
            p   = tgt_probs[0, i + 1, tok].item()
            q   = draft_probs_list[i]
            if torch.rand(1).item() <= min(1.0, p / (q + 1e-8)):
                generated  = torch.cat([generated, draft_seq[:, i:i+1]], dim=-1)
                tokens_gen += 1
            else:
                corrected = F.relu(tgt_probs[0, i + 1] - tgt_probs[0, i])
                mass = corrected.sum()
                if mass < 1e-6:
                    resampled = torch.multinomial(tgt_probs[0, i + 1], 1)
                else:
                    resampled = torch.multinomial(corrected / mass, 1)
                generated  = torch.cat([generated, resampled.unsqueeze(0)], dim=-1)
                tokens_gen += 1
                break

    return tokenizer_e.decode(generated[0][input_ids.shape[1]:], skip_special_tokens=True)


he_c4   = run_humaneval(generate_eagle, "eagle2")
mbpp_c4 = run_mbpp(generate_eagle,      "eagle2")
save_condition_results("eagle2", he_c4, mbpp_c4)

del base_e, draft_e
torch.cuda.empty_cache()

## 8. Final Summary Table — All 4 Conditions

In [ ]:
import pandas as pd

all_results = {
    "baseline_generic": (he_c1, mbpp_c1),
    "domain_tuned":     (he_c2, mbpp_c2),
    "medusa":           (he_c3, mbpp_c3),
    "eagle2":           (he_c4, mbpp_c4),
}

rows = []
for condition, (he, mb) in all_results.items():
    rows.append({
        "Condition":       condition,
        "HumanEval P@1":  f"{he['metrics']['pass@1']*100:.1f}%",
        "HumanEval P@10": f"{he['metrics']['pass@10']*100:.1f}%",
        "MBPP P@1":       f"{mb['metrics']['pass@1']*100:.1f}%",
        "MBPP P@10":      f"{mb['metrics']['pass@10']*100:.1f}%",
    })

df = pd.DataFrame(rows)
print("\n" + "="*65)
print(df.to_string(index=False))
print("="*65)

# Save combined summary
summary = {c: {"humaneval": he["metrics"], "mbpp": mb["metrics"]}
           for c, (he, mb) in all_results.items()}
with open(os.path.join(RESULTS_DIR, "benchmark_summary_all_conditions.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nSaved benchmark_summary_all_conditions.json")

In [ ]:
import matplotlib.pyplot as plt

conditions = list(all_results.keys())
he_p1  = [all_results[c][0]["metrics"]["pass@1"]  for c in conditions]
he_p10 = [all_results[c][0]["metrics"]["pass@10"] for c in conditions]
mb_p1  = [all_results[c][1]["metrics"]["pass@1"]  for c in conditions]
mb_p10 = [all_results[c][1]["metrics"]["pass@10"] for c in conditions]

x = range(len(conditions))
w = 0.2
colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, p1, p10, title in zip(axes, [he_p1, mb_p1], [he_p10, mb_p10],
                               ["HumanEval", "MBPP"]):
    bars1 = ax.bar([i - w/2 for i in x], p1,  width=w, label="Pass@1",  color=colors, alpha=0.9)
    bars2 = ax.bar([i + w/2 for i in x], p10, width=w, label="Pass@10", color=colors, alpha=0.5, hatch="//")
    ax.set_xticks(list(x))
    ax.set_xticklabels([c.replace("_", "\n") for c in conditions], fontsize=9)
    ax.set_ylabel("Pass Rate")
    ax.set_title(f"{title}: Pass@1 & Pass@10 — All 4 Conditions")
    ax.legend()
    ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "benchmark_passk_all_conditions.png"), dpi=150)
plt.show()
print("Saved benchmark_passk_all_conditions.png")